# Bốn loại chart trên Bike Sharing: line, bar, heatmap, stacked

Bộ `bike-sharing-uci`: lượt thuê xe đạp Capital Bikeshare (Washington DC) theo **từng giờ**, 2011–2012
(`hour.csv`, 17 379 dòng). Mỗi dòng là một giờ: `dteday` (ngày), `season` (1–4), `mnth`, `hr` (0–23),
`weekday` (0 = Chủ nhật … 6 = Thứ Bảy), `casual` (khách vãng lai), `registered` (thành viên),
`cnt = casual + registered`.

Bạn là DA của công ty xe đạp. Bốn người hỏi bốn câu, mỗi câu một loại chart:

| Người hỏi | Câu hỏi | Loại câu hỏi | Chart |
| --- | --- | --- | --- |
| CEO | Lượt thuê mỗi ngày đi lên hay đi xuống trong hai năm? | Xu hướng | Line |
| Trưởng vận hành | Mùa nào cần nhiều xe nhất? | So sánh | Bar |
| Trưởng điều phối | Giờ nào, thứ nào cần bù xe? | Hai chiều phân loại | Heatmap |
| Marketing | Khách vãng lai chiếm bao nhiêu phần, đổi theo mùa ra sao? | Thành phần theo thời gian | Stacked bar 100% |

Việc của bạn: viết 4 hàm chuẩn bị dữ liệu (cell chấm điểm), vẽ 4 chart (cell `explore`), đọc từng chart
theo quy trình 5 bước của lý thuyết, rồi **làm phần trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm/hằng. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/08-visualization-chart-reading/exercises/basic/02-bike-chart-types -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc hour.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
hour = pd.read_csv(BIKE / "hour.csv")
print(hour.shape)
hour.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Nhãn mùa theo tệp Readme.txt của UCI. Đọc kỹ phần 3 trước khi tin nhãn này.
SEASON_LABELS = {1: "spring", 2: "summer", 3: "fall", 4: "winter"}
WEEKDAY_VI = ["CN", "T2", "T3", "T4", "T5", "T6", "T7"]   # weekday 0 = Chủ nhật

## 1. Xu hướng: line chart lượt thuê mỗi ngày

`daily_rides(hour) -> pd.Series`: tổng `cnt` mỗi ngày. Index là `DatetimeIndex` (đổi `dteday` sang
datetime, vì `read_csv` đọc nó thành chuỗi), sắp tăng dần theo ngày.

Trục x là thời gian thì vẽ line. 731 điểm theo ngày rất nhiễu (thứ, thời tiết), nên vẽ thêm đường
trung bình trượt 28 ngày để mắt thấy xu hướng, giữ đường ngày mờ phía sau.

In [ ]:
def daily_rides(hour: pd.DataFrame) -> pd.Series:
    """Tổng cnt mỗi ngày, index là DatetimeIndex tăng dần."""
    # TODO: pd.to_datetime(hour["dteday"]), groupby theo ngày, sum cnt
    raise NotImplementedError

In [ ]:
# explore: line chart theo ngày + trung bình trượt 28 ngày
daily = daily_rides(hour)
roll = daily.rolling(28, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily.index, daily.values, color="#c9ced6", lw=0.8, label="mỗi ngày")
ax.plot(roll.index, roll.values, color="#4c78a8", lw=2, label="trung bình 28 ngày")
ax.set_ylim(0, None)
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylabel("lượt thuê/ngày")
ax.legend(frameon=False, loc="upper left")
by_year = daily.groupby(daily.index.year).sum()
ax.set_title(f"Lượt thuê 2012 cao hơn 2011 {by_year[2012] / by_year[2011] - 1:.0%}; mùa đông năm nào cũng chững",
             loc="left")
plt.show()
print(by_year)
print("ngày cao nhất:", daily.idxmax().date(), daily.max(), "| thấp nhất:", daily.idxmin().date(), daily.min())

## 2. So sánh: bar chart theo mùa

`rides_by_season(hour) -> pd.Series`: tổng `cnt` theo mùa, index là nhãn chữ trong `SEASON_LABELS`
(`spring`, `summer`, `fall`, `winter`), **sắp giảm dần**: đây là bảng xếp hạng, trả lời "mùa nào cao nhất".

Lúc vẽ thì khác: mùa là category **có thứ tự tự nhiên** (vòng tròn của năm), nên bar giữ thứ tự mùa
chứ không sắp theo giá trị. Hàm trả bảng xếp hạng, chart dùng `reindex` về thứ tự tự nhiên.

In [ ]:
def rides_by_season(hour: pd.DataFrame) -> pd.Series:
    """Tổng cnt theo mùa, index là nhãn trong SEASON_LABELS, sắp giảm dần."""
    # TODO: groupby season, sum cnt, đổi index bằng SEASON_LABELS, sort_values giảm dần
    raise NotImplementedError

In [ ]:
# explore: bar giữ thứ tự mùa, nhấn mùa cao nhất
season = rides_by_season(hour)
print(season)
natural = season.reindex(list(SEASON_LABELS.values()))
colors = ["#f58518" if v == natural.max() else "#c9ced6" for v in natural.values]

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(natural.index, natural.values / 1e6, color=colors)
ax.bar_label(bars, labels=[f"{v / 1e6:.2f}M" for v in natural.values], padding=3)
ax.set_ylim(0, natural.max() / 1e6 * 1.15)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_yticks([])
ax.set_title(f"'{season.index[0]}' đông nhất, gấp {season.iloc[0] / season.iloc[-1]:.1f} lần '{season.index[-1]}'", loc="left")
plt.show()

## 3. Bước 1 của quy trình đọc chart: nhãn có đúng không?

Chart bar ở trên dùng nhãn `SEASON_LABELS` lấy từ Readme của UCI. Trước khi đưa cho trưởng vận hành,
kiểm xem mỗi mã `season` thật ra gồm những ngày nào (cell dưới). Đối chiếu với lịch: ở Washington DC,
tháng 1 là mùa gì? Ghi nhận xét của bạn; câu trắc nghiệm `season-label` hỏi đúng điều này.

In [ ]:
# explore: năm 2011, mỗi mã season gồm những ngày nào của những tháng nào
y2011 = hour[hour["yr"] == 0].assign(dteday=pd.to_datetime(hour["dteday"]))
check = y2011.groupby(["season", "mnth"])["dteday"].agg(tu="min", den="max")
check["so_ngay"] = y2011.groupby(["season", "mnth"])["dteday"].nunique()
check.rename(index=SEASON_LABELS, level="season")

## 4. Hai chiều phân loại: heatmap thứ × giờ

`weekday_hour_matrix(hour) -> pd.DataFrame`: ma trận **7 × 24**, hàng là `weekday` 0–6, cột là `hr`
0–23, giá trị là **trung bình** `cnt` mỗi giờ (không phải tổng: số thứ Hai và số thứ Bảy trong hai năm
không bằng nhau, và một số giờ không có dòng nào). Ô không có dữ liệu để `NaN`, không điền 0. Luôn trả
đủ 7 hàng và 24 cột, kể cả khi dữ liệu thiếu thứ hoặc giờ.

Heatmap tốt để thấy **pattern**; khi cần so hai ô cụ thể, đọc số trong ma trận chứ đừng so màu.

In [ ]:
def weekday_hour_matrix(hour: pd.DataFrame) -> pd.DataFrame:
    """Ma trận 7 × 24 (weekday × hr) của trung bình cnt; ô không có dữ liệu là NaN."""
    # TODO: pivot_table(index="weekday", columns="hr", values="cnt", aggfunc="mean")
    # TODO: reindex đủ 7 hàng (0..6) và 24 cột (0..23)
    raise NotImplementedError

In [ ]:
# explore: heatmap, hàng xếp T2 → CN để cuối tuần nằm cạnh nhau
heat = weekday_hour_matrix(hour)
show = heat.reindex([1, 2, 3, 4, 5, 6, 0])
show.index = [WEEKDAY_VI[i] for i in show.index]

fig, ax = plt.subplots(figsize=(12, 3.8))
sns.heatmap(show, ax=ax, cmap="viridis", cbar_kws={"label": "lượt/giờ (trung bình)"})
ax.set(xlabel="giờ trong ngày", ylabel="")
ax.set_title("Ngày làm việc có hai đỉnh đi làm; cuối tuần đông giữa trưa", loc="left")
plt.show()
print("giờ đông nhất của từng thứ:", {WEEKDAY_VI[d]: int(h) for d, h in heat.idxmax(axis=1).items()})

## 5. Thành phần theo thời gian: stacked bar 100%

`casual_share_by_month(hour) -> pd.DataFrame`: mỗi tháng lịch (2011-01 … 2012-12) một dòng, index là
`DatetimeIndex` ngày đầu tháng, hai cột `casual`, `registered` là **tỉ trọng** trong tổng lượt của tháng
(mỗi dòng cộng lại bằng 1).

Câu hỏi của marketing là tỉ trọng, không phải quy mô, nên vẽ stacked **100%**. Stacked tuyệt đối sẽ để
mùa vụ của tổng lượt thuê lấn át thay đổi về cơ cấu.

In [ ]:
def casual_share_by_month(hour: pd.DataFrame) -> pd.DataFrame:
    """Tỉ trọng casual/registered mỗi tháng lịch; index là ngày đầu tháng, mỗi dòng cộng lại bằng 1."""
    # TODO: tháng = pd.to_datetime(dteday) rồi .dt.to_period("M").dt.to_timestamp()
    # TODO: tổng casual, registered theo tháng; chia cho tổng của dòng
    raise NotImplementedError

In [ ]:
# explore: stacked bar 100%, casual ở dưới để đọc được từ trục 0
from matplotlib.ticker import PercentFormatter

share = casual_share_by_month(hour)
labels = share.index.strftime("%m/%y")

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(labels, share.casual, color="#f58518", label="casual")
ax.bar(labels, share.registered, bottom=share.casual, color="#c9ced6", label="registered")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncols=2, loc="upper left", bbox_to_anchor=(0, 1.12))
ax.tick_params(axis="x", labelrotation=90)
top = share.casual.idxmax()
ax.set_title(f"Khách vãng lai lên tới {share.casual.max():.0%} lượt vào {top:%m/%Y}; mùa đông chỉ còn khoảng 10%",
             loc="left", pad=28)
plt.show()
print(share.casual.round(3).to_string())

## 6. Đọc chart, rồi làm trắc nghiệm

Với mỗi chart, viết ra (vào cell này) theo quy trình 5 bước: trục và đơn vị (tổng hay trung bình? kỳ
nào?), hình dạng, điểm bất thường, so với tham chiếu, câu hỏi tiếp theo. Tách **quan sát** khỏi
**kết luận**.

Gợi ý câu hỏi tiếp theo: đỉnh tỉ trọng casual trùng mùa du lịch hay trùng tháng có nhiều ngày lễ?
Ngày thấp nhất (22 lượt) là lỗi dữ liệu hay sự kiện thật? Cần breakdown nào để biết?

Xong thì mở app, làm phần **trắc nghiệm** của bài.